# Phase 3 - XAI map generation

This notebook loads the Phase 2 ResNet-50 checkpoint and generates Grad-CAM, SHAP and LIME explanation maps. The small sampled set is used for qualitative visual inspection. The final export section creates `.npy` maps and a manifest CSV for Phase 4 quantitative comparison with lesion masks and pseudo-concept maps.

## Environment note

In [ ]:
# Required packages: captum, shap, lime, scikit-image.
# Prefer installing these from requirements.txt rather than from inside the notebook.
# Uncomment only in a fresh local/Colab environment.
# import subprocess
# subprocess.run(["pip", "install", "captum", "shap", "lime", "scikit-image"], check=True)

## Imports and config

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# XAI
from captum.attr import LayerGradCam, LayerAttribution
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths — same structure as Phase 2
BASE_DIR       = Path("..")
HAM_IMG_DIR    = BASE_DIR / "data" / "ham10000" / "images"
HAM_CSV        = BASE_DIR / "data" / "preprocessed_manifests" / "ham10000_preprocessed.csv"
CHECKPOINT_DIR = Path("checkpoints")
XAI_OUT_DIR    = Path("xai_outputs")
XAI_MAP_DIR    = XAI_OUT_DIR / "maps"
XAI_FIG_DIR    = XAI_OUT_DIR / "figures"
XAI_MANIFEST_DIR = XAI_OUT_DIR / "manifests"
for d in [XAI_MAP_DIR, XAI_FIG_DIR, XAI_MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Hyperparameters (must match Phase 2)
IMG_SIZE   = 224
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_PER_CLASS = 3   # qualitative examples per lesion class
BEST_THRESHOLD = 0.85  # from Phase 2 validation threshold tuning
GRADCAM_LAYER = "layer3"  # layer3 gives finer localisation than layer4 for this task

print(f"Using device: {DEVICE}")

## Rebuild model and load checkpoint

In [ ]:
# Rebuild the exact same architecture as Phase 2
model = models.resnet50(weights=None)

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 1)
)

CHECKPOINT_PATH = CHECKPOINT_DIR / "best_resnet50_finetuned.pt"
assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH.resolve()}"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Checkpoint loaded: {CHECKPOINT_PATH}")
print(f"Validation accuracy at save: {checkpoint.get('val_acc', float('nan')):.4f}")

## Load data and build val split (same as Phase 2)

In [ ]:
from sklearn.model_selection import train_test_split

KEEP_COLS = ['stem', 'dx', 'label', 'label_name', 'image_exists']
df = pd.read_csv(HAM_CSV)[KEEP_COLS].copy()
df = df[df['image_exists'] == True].reset_index(drop=True)

binary_map = {'benign': 0, 'malignant': 1}
df['binary_label'] = df['label_name'].map(binary_map)
df = df[df['binary_label'].notna()].reset_index(drop=True)
df['binary_label'] = df['binary_label'].astype(int)

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['binary_label'], random_state=SEED
)
val_df = val_df.reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")
print("Validation dx distribution:")
print(val_df['dx'].value_counts())
print("\nValidation label distribution:")
print(val_df['label_name'].value_counts())

## Transforms and helper functions

In [ ]:
# Validation transform — no augmentation (same as Phase 2)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ImageNet mean/std for denormalisation (needed for display)
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    """Convert normalised tensor → displayable numpy RGB image."""
    img = tensor.detach().cpu().numpy().transpose(1, 2, 0)  # ← add .detach()
    img = (img * STD + MEAN)
    return np.clip(img, 0, 1)

def load_image_tensor(img_path):
    """Load a single image as a (1, C, H, W) tensor on DEVICE."""
    img = Image.open(img_path).convert("RGB")
    return val_transform(img).unsqueeze(0).to(DEVICE)

def predict_prob(tensor):
    """Return malignant probability for a (1,C,H,W) tensor."""
    with torch.no_grad():
        logit = model(tensor)
        return torch.sigmoid(logit).item()

## Sample N images per lesion class

In [ ]:
# HAM10000 dx classes
DX_CLASSES = sorted(val_df['dx'].unique())
print("Classes:", DX_CLASSES)

# Sample N_PER_CLASS images from each dx class
samples = []
for dx in DX_CLASSES:
    subset = val_df[val_df['dx'] == dx]
    chosen = subset.sample(min(N_PER_CLASS, len(subset)), random_state=SEED)
    samples.append(chosen)

samples_df = pd.concat(samples).reset_index(drop=True)
print(f"\nTotal samples selected: {len(samples_df)}")
print(samples_df[['stem', 'dx', 'label_name']].to_string(index=False))

## Grad-CAM

In [ ]:
from captum.attr import LayerGradCam, LayerAttribution

# Target layer for Grad-CAM.
# layer3 usually gives finer spatial localisation than layer4 on 224 x 224 dermoscopy images.
target_layer = model.layer3[-1] if GRADCAM_LAYER == "layer3" else model.layer4[-1]
gradcam = LayerGradCam(model, target_layer)
print(f"Grad-CAM target layer: {GRADCAM_LAYER}")

def get_gradcam(img_tensor):
    """
    Returns a (224, 224) numpy heatmap for a single (1,C,H,W) tensor.
    target=0 because our model outputs a single logit (binary).
    """
    img_tensor = img_tensor.requires_grad_(True)
    attr = gradcam.attribute(img_tensor, target=0)
    # Upsample to input size
    attr_up = LayerAttribution.interpolate(attr, (IMG_SIZE, IMG_SIZE))
    heatmap  = attr_up.squeeze().cpu().detach().numpy()
    # Normalise to [0, 1]
    heatmap  = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap

# Quick test
test_row  = samples_df.iloc[0]
test_path = HAM_IMG_DIR / f"{test_row['stem']}.jpg"
test_t    = load_image_tensor(test_path)
heatmap   = get_gradcam(test_t)

plt.figure(figsize=(5, 4))
plt.imshow(denormalize(test_t.squeeze(0)))
plt.imshow(heatmap, cmap='jet', alpha=0.45)
plt.title(f"Grad-CAM test — {test_row['dx']} ({test_row['label_name']})")
plt.axis('off')
plt.tight_layout()
plt.show()
print("Grad-CAM working ✓")

## SHAP

In [ ]:
# ── Build a small background set for SHAP (50 val images) ─────────────────
bg_stems = val_df.sample(50, random_state=SEED)['stem'].tolist()
bg_tensors = torch.cat([
    load_image_tensor(HAM_IMG_DIR / f"{s}.jpg") for s in bg_stems
], dim=0)  # (50, C, H, W)

# Wrapper: SHAP needs a function that takes numpy (N,H,W,C) → numpy (N,1)
def model_predict_shap(images_np):
    """images_np: (N, H, W, C) float32 numpy, pixel values in [0,1]."""
    tensors = []
    for img in images_np:
        t = transforms.ToTensor()(img)
        t = transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225])(t)
        tensors.append(t)
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        logits = model(batch)
        probs  = torch.sigmoid(logits).cpu().numpy()
    return probs  # (N, 1)

explainer_shap = shap.GradientExplainer(
    model,
    bg_tensors
)

def get_shap(img_tensor):
    """
    Returns a (224, 224) SHAP saliency map (absolute mean across channels).
    """
    shap_values = explainer_shap.shap_values(img_tensor)
    # shap_values shape: (1, C, H, W)
    sv = np.abs(shap_values[0])   # (C, H, W)
    sv = sv.mean(axis=0)           # (H, W) — average across RGB
    if sv.max() > 0:
        sv /= sv.max()
    return sv

# Quick test
sv_map = get_shap(test_t)
plt.figure(figsize=(5, 4))
plt.imshow(denormalize(test_t.squeeze(0)))
plt.imshow(sv_map, cmap='hot', alpha=0.45)
plt.title(f"SHAP test — {test_row['dx']} ({test_row['label_name']})")
plt.axis('off')
plt.tight_layout()
plt.show()
print("SHAP working ✓")

## LIME

In [ ]:
explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)

def get_lime(img_path):
    """
    Returns (explanation, img_np) where img_np is the raw (H,W,3) uint8 array.
    LIME works on raw pixel images, not normalised tensors.
    """
    img_np = np.array(
        Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    )  # uint8 (H, W, 3)

    def batch_predict(images):
        """images: list of (H,W,3) uint8 numpy arrays."""
        return model_predict_shap(
            np.array(images).astype(np.float32) / 255.0
        )

    explanation = explainer_lime.explain_instance(
        img_np,
        batch_predict,
        top_labels=1,
        hide_color=0,
        num_samples=500,    # increase to 1000+ for better quality, slower
        random_seed=SEED
    )
    return explanation, img_np

def lime_heatmap(explanation, img_np):
    """Convert LIME explanation → normalised (H,W) heatmap array."""
    top_label = explanation.top_labels[0]
    # Get positive superpixel contributions
    temp, mask = explanation.get_image_and_mask(
        top_label,
        positive_only=True,
        num_features=10,
        hide_rest=False
    )
    heatmap = explanation.get_image_and_mask(
        top_label,
        positive_only=False,
        num_features=10,
        hide_rest=False
    )[1].astype(float)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap

# Quick test
lime_exp, img_np = get_lime(test_path)
lm = lime_heatmap(lime_exp, img_np)
plt.figure(figsize=(5, 4))
plt.imshow(img_np)
plt.imshow(lm, cmap='RdYlGn', alpha=0.45)
plt.title(f"LIME test — {test_row['dx']} ({test_row['label_name']})")
plt.axis('off')
plt.tight_layout()
plt.show()
print("LIME working ✓")

## Generate and save maps for the qualitative visual subset

In [ ]:
# ── Generate Grad-CAM, SHAP, LIME for every sample and save arrays ─────────
results = []  # will hold dicts for the visualisation cell

print(f"Processing {len(samples_df)} images...\n")

for idx, row in samples_df.iterrows():
    img_path   = HAM_IMG_DIR / f"{row['stem']}.jpg"
    img_tensor = load_image_tensor(img_path)
    prob       = predict_prob(img_tensor)

    # Grad-CAM
    gc_map = get_gradcam(img_tensor)

    # SHAP
    sv_map = get_shap(img_tensor)

    # LIME
    lime_exp, img_np = get_lime(img_path)
    lm_map = lime_heatmap(lime_exp, img_np)

    # Save maps as .npy arrays
    stem = row['stem']
    gradcam_path = XAI_MAP_DIR / f"{stem}_gradcam_{GRADCAM_LAYER}.npy"
    shap_path = XAI_MAP_DIR / f"{stem}_shap.npy"
    lime_path = XAI_MAP_DIR / f"{stem}_lime.npy"

    np.save(gradcam_path, gc_map)
    np.save(shap_path, sv_map)
    np.save(lime_path, lm_map)

    results.append({
        'stem':       stem,
        'dx':         row['dx'],
        'label_name': row['label_name'],
        'prob':       prob,
        'pred_label': int(prob >= BEST_THRESHOLD),
        'gradcam_path': str(gradcam_path),
        'shap_path': str(shap_path),
        'lime_path': str(lime_path),
        'img_tensor': img_tensor,
        'img_np':     img_np,
        'gradcam':    gc_map,
        'shap':       sv_map,
        'lime':       lm_map,
    })

    print(f"  [{idx+1:02d}/{len(samples_df)}] {stem} | {row['dx']:4s} | "
          f"{row['label_name']:10s} | p(malignant)={prob:.3f}")

qual_manifest = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["img_tensor", "img_np", "gradcam", "shap", "lime"]}
    for r in results
])
qual_manifest_path = XAI_MANIFEST_DIR / "xai_visual_subset_manifest.csv"
qual_manifest.to_csv(qual_manifest_path, index=False)

print(f"\nDone. Arrays saved to {XAI_MAP_DIR.resolve()}")
print(f"Qualitative manifest saved to: {qual_manifest_path.resolve()}")

## Visualisation grid (Image | Grad-CAM | SHAP | LIME)

In [ ]:
# ── One figure per dx class — 3 rows (samples) × 4 columns (methods) ──────
CMAPS = {'gradcam': 'jet', 'shap': 'hot', 'lime': 'RdYlGn'}

for dx in DX_CLASSES:
    dx_results = [r for r in results if r['dx'] == dx]
    n = len(dx_results)

    fig, axes = plt.subplots(n, 4, figsize=(16, 4.5 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)

    fig.suptitle(f"XAI Visualisations — {dx.upper()}  "
                 f"({'malignant' if dx_results[0]['label_name']=='malignant' else 'benign'})",
                 fontsize=14, fontweight='bold', y=1.01)

    col_titles = ['Original', 'Grad-CAM', 'SHAP', 'LIME']
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight='bold')

    for row_i, r in enumerate(dx_results):
        orig = denormalize(r['img_tensor'].squeeze(0))

        # Column 0 — original image
        axes[row_i, 0].imshow(orig)
        axes[row_i, 0].set_ylabel(
            f"p={r['prob']:.2f}", fontsize=9, rotation=0,
            labelpad=50, va='center'
        )

        # Columns 1-3 — XAI overlays
        for col_i, (method, cmap) in enumerate(CMAPS.items(), start=1):
            axes[row_i, col_i].imshow(orig)
            axes[row_i, col_i].imshow(
                r[method], cmap=cmap, alpha=0.5,
                vmin=0, vmax=1
            )

        for ax in axes[row_i]:
            ax.axis('off')

    plt.tight_layout()
    fig.savefig(XAI_FIG_DIR / f"xai_grid_{dx}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: xai_grid_{dx}.png")

print(f"\nAll grids saved to {XAI_FIG_DIR.resolve()}")

## Combined summary figure (report-ready)

In [ ]:
# ── Single figure showing 1 example per class — good for the report ────────
fig, axes = plt.subplots(len(DX_CLASSES), 4, figsize=(16, 4 * len(DX_CLASSES)))

col_titles = ['Original', 'Grad-CAM', 'SHAP', 'LIME']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight='bold')

for row_i, dx in enumerate(DX_CLASSES):
    r    = next(res for res in results if res['dx'] == dx)
    orig = denormalize(r['img_tensor'].squeeze(0))

    axes[row_i, 0].imshow(orig)
    axes[row_i, 0].set_ylabel(
        f"{dx}\np={r['prob']:.2f}",
        fontsize=9, rotation=0, labelpad=55, va='center'
    )

    for col_i, (method, cmap) in enumerate(CMAPS.items(), start=1):
        axes[row_i, col_i].imshow(orig)
        axes[row_i, col_i].imshow(r[method], cmap=cmap, alpha=0.5, vmin=0, vmax=1)

    for ax in axes[row_i]:
        ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.suptitle('XAI Summary — One example per lesion class',
             fontsize=14, fontweight='bold', y=0.99)
fig.savefig(XAI_FIG_DIR / "xai_summary_report.png", dpi=200, bbox_inches='tight')
plt.show()
print("Report figure saved: xai_summary_report.png")

## Phase 3B - Comprehensive XAI export for Phase 4

This section is intentionally placed at the end. It can be switched on when qualitative checks are satisfactory. It exports XAI maps for the full validation split and writes `xai_outputs_manifest.csv`, which Phase 4 can merge with pseudo-concept manifests by `stem`.

In [ ]:
RUN_FULL_XAI_EXPORT = False  # Set True when ready; SHAP and LIME can be slow.

if RUN_FULL_XAI_EXPORT:
    xai_eval_df = val_df.copy().reset_index(drop=True)
    manifest_rows = []

    print(f"Processing full validation set: {len(xai_eval_df)} images")

    for idx, row in xai_eval_df.iterrows():
        img_path = HAM_IMG_DIR / f"{row['stem']}.jpg"
        img_tensor = load_image_tensor(img_path)
        prob = predict_prob(img_tensor)

        gc_map = get_gradcam(img_tensor)
        sv_map = get_shap(img_tensor)
        lime_exp, img_np = get_lime(img_path)
        lm_map = lime_heatmap(lime_exp, img_np)

        stem = row["stem"]
        gradcam_path = XAI_MAP_DIR / f"{stem}_gradcam_{GRADCAM_LAYER}.npy"
        shap_path = XAI_MAP_DIR / f"{stem}_shap.npy"
        lime_path = XAI_MAP_DIR / f"{stem}_lime.npy"

        np.save(gradcam_path, gc_map)
        np.save(shap_path, sv_map)
        np.save(lime_path, lm_map)

        manifest_rows.append({
            "stem": stem,
            "dx": row["dx"],
            "label_name": row["label_name"],
            "binary_label": int(row["binary_label"]),
            "prob_malignant": float(prob),
            "pred_label": int(prob >= BEST_THRESHOLD),
            "threshold": float(BEST_THRESHOLD),
            "gradcam_layer": GRADCAM_LAYER,
            "image_path": str(img_path),
            "gradcam_path": str(gradcam_path),
            "shap_path": str(shap_path),
            "lime_path": str(lime_path),
        })

        if (idx + 1) % 25 == 0 or idx == 0:
            print(f"[{idx+1:04d}/{len(xai_eval_df)}] {stem} | p(malignant)={prob:.3f}")

    xai_manifest = pd.DataFrame(manifest_rows)
    manifest_path = XAI_MANIFEST_DIR / "xai_outputs_manifest.csv"
    xai_manifest.to_csv(manifest_path, index=False)

    print(f"Saved full XAI manifest: {manifest_path.resolve()}")
    display(xai_manifest.head())
else:
    print("Full XAI export skipped. Set RUN_FULL_XAI_EXPORT=True when ready.")